### Miisng Values Imputation using Bayesian Diffusion Transformer

In [ ]:

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

# Define your dataset
class TabularTimeSeriesDataset(Dataset):
    def __init__(self, df, label_cols, demographic_cols):
        self.label_cols = label_cols
        self.demographic_cols = demographic_cols

        # Normalize Age column
        if 'Age' in demographic_cols and 'Age' in df.columns:
            df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
            df['Age'] = (df['Age'] - df['Age'].mean()) / df['Age'].std()

        # One-hot encode 'Sex' if in demographics
        if 'Sex' in demographic_cols and 'Sex' in df.columns:
            df = pd.get_dummies(df, columns=['Sex'])

        # Fill missing values with 0
        df.fillna(0, inplace=True)

        # Combine label and demographic columns
        selected_cols = label_cols + [col for col in df.columns if col in demographic_cols or col.startswith('Sex_')]

        self.data = df[selected_cols].copy()
        self.data = self.data.apply(pd.to_numeric, errors="coerce")
        self.data = self.data.fillna(0).astype(np.float32)

        # Create a binary mask for labels only
        self.mask = (~df[label_cols].isna()).astype(np.float32).values

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = torch.tensor(self.data.iloc[idx].values, dtype=torch.float32)
        mask = torch.tensor(self.mask[idx], dtype=torch.float32)
        return x, mask

# DIFFUSION TRANSFORMER MODEL
class BayesianDiffusionTransformer(nn.Module):
    def __init__(self, input_dim, timesteps=1000):
        super().__init__()
        self.timesteps = timesteps
        self.input_dim = input_dim
        self.fc = nn.Sequential(
            nn.Linear(input_dim + 1, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )
        self.beta = torch.linspace(1e-4, 0.02, timesteps)
        self.alpha = 1. - self.beta
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)

    def forward(self, x, t):
        t = t.unsqueeze(1).float() / self.timesteps  # normalize timestep
        xt = torch.cat([x, t], dim=1)
        return self.fc(xt)

    def q_sample(self, x_start, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_start)
        alpha_hat_t = self.alpha_hat[t].unsqueeze(1).to(x_start.device)
        return torch.sqrt(alpha_hat_t) * x_start + torch.sqrt(1 - alpha_hat_t) * noise
    
import os
from sklearn.metrics import mean_absolute_error, r2_score

import os
import torch
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

# Directory to save model checkpoints
SAVE_DIR = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/Missing_values_Imputation_Beyesian_Diffusion_Transformer/"
os.makedirs(SAVE_DIR, exist_ok=True)

def save_checkpoint(model, optimizer, epoch, filename):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, filename)

def load_checkpoint(model, optimizer, filename, device='cpu'):
    checkpoint = torch.load(filename, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return checkpoint['epoch']

def train(model, dataloader, epochs=300, device='cpu', resume_path="model_epoch_100.pt"):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    start_epoch = 0

    # Modified to construct full resume path using SAVE_DIR
    resume_path = os.path.join(SAVE_DIR, resume_path) if resume_path else None

    if resume_path and os.path.exists(resume_path):
        print(f"Resuming training from {resume_path}")
        start_epoch = load_checkpoint(model, optimizer, resume_path, device)
        print(f"Resumed from epoch {start_epoch}")
    else:
        print(f"No checkpoint found at {resume_path}, starting from scratch.")

    for epoch in range(start_epoch, epochs):
        model.train()
        total_loss = 0
        all_preds = []
        all_targets = []

        for x, mask in dataloader:
            x = x.to(device)
            mask = mask.to(device)
            t = torch.randint(0, model.timesteps, (x.size(0),), device=device).long()
            noise = torch.randn_like(x)
            x_noisy = model.q_sample(x, t, noise)
            pred_noise = model(x_noisy, t)

            loss = ((pred_noise[:, :mask.shape[1]] - noise[:, :mask.shape[1]]) ** 2 * mask).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            all_preds.append(pred_noise[:, :mask.shape[1]].detach().cpu().numpy())
            all_targets.append(noise[:, :mask.shape[1]].cpu().numpy())

        # Evaluation metrics
        preds = np.vstack(all_preds)
        targets = np.vstack(all_targets)
        mae = np.mean(np.abs(preds - targets))
        ss_res = np.sum((targets - preds) ** 2)
        ss_tot = np.sum((targets - np.mean(targets)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot != 0 else 0.0

        print(f"Epoch {epoch + 1}, Loss: {total_loss:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")

        # Save model
        checkpoint_path = os.path.join(SAVE_DIR, f"model_epoch_{epoch+1}.pt")
        save_checkpoint(model, optimizer, epoch+1, checkpoint_path)

# Load data and run training
df = pd.read_csv(csv_path)
csv_path = "/mnt/Internal/MedImage/merged_dataset-Copy1.csv"
label_cols = ['Cardiomegaly', 'Edema', 'Consolidation', 'Pneumonia']
demographic_cols = ['Age', 'Sex']

df = pd.read_csv(csv_path)
dataset = TabularTimeSeriesDataset(df, label_cols, demographic_cols)
input_dim = dataset.data.shape[1]
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

model = BayesianDiffusionTransformer(input_dim=input_dim)
train(model, dataloader)

In [3]:
train_df_full = pd.read_csv("/mnt/Internal/MedImage/merged_dataset-Copy1.csv")

train_df_full = pd.get_dummies(train_df_full, columns=["GENDER", "PRIMARY_RACE", "ETHNICITY"])
train_df_full.columns.tolist()

# Convert categorical columns to integers in train_df_full

# Race
train_df_full['PRIMARY_RACE_American Indian or Alaska Native'] = train_df_full['PRIMARY_RACE_American Indian or Alaska Native'].astype(int)
train_df_full['PRIMARY_RACE_Asian'] = train_df_full['PRIMARY_RACE_Asian'].astype(int)
#train_df_full['PRIMARY_RACE_Asian - Historical Conv'] = train_df_full['PRIMARY_RACE_Asian - Historical Conv'].astype(int)
train_df_full['PRIMARY_RACE_Asian, Hispanic'] = train_df_full['PRIMARY_RACE_Asian, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Asian, non-Hispanic'] = train_df_full['PRIMARY_RACE_Asian, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Black or African American'] = train_df_full['PRIMARY_RACE_Black or African American'].astype(int)
train_df_full['PRIMARY_RACE_Black, Hispanic'] = train_df_full['PRIMARY_RACE_Black, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Black, non-Hispanic'] = train_df_full['PRIMARY_RACE_Black, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Native American, Hispanic'] = train_df_full['PRIMARY_RACE_Native American, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Native American, non-Hispanic'] = train_df_full['PRIMARY_RACE_Native American, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Native Hawaiian or Other Pacific Islander'] = train_df_full['PRIMARY_RACE_Native Hawaiian or Other Pacific Islander'].astype(int)
#train_df_full['PRIMARY_RACE_Other'] = train_df_full['PRIMARY_RACE_Other'].astype(int)
#train_df_full['PRIMARY_RACE_Other, Hispanic'] = train_df_full['PRIMARY_RACE_Other, Hispanic'].astype(int)
#train_df_full['PRIMARY_RACE_Other, non-Hispanic'] = train_df_full['PRIMARY_RACE_Other, non-Hispanic'].astype(int)
#train_df_full['PRIMARY_RACE_Pacific Islander, Hispanic'] = train_df_full['PRIMARY_RACE_Pacific Islander, Hispanic'].astype(int)
#train_df_full['PRIMARY_RACE_Pacific Islander, non-Hispanic'] = train_df_full['PRIMARY_RACE_Pacific Islander, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Patient Refused'] = train_df_full['PRIMARY_RACE_Patient Refused'].astype(int)
train_df_full['PRIMARY_RACE_Race and Ethnicity Unknown'] = train_df_full['PRIMARY_RACE_Race and Ethnicity Unknown'].astype(int)
train_df_full['PRIMARY_RACE_Unknown'] = train_df_full['PRIMARY_RACE_Unknown'].astype(int)
train_df_full['PRIMARY_RACE_White'] = train_df_full['PRIMARY_RACE_White'].astype(int)
train_df_full['PRIMARY_RACE_White or Caucasian'] = train_df_full['PRIMARY_RACE_White or Caucasian'].astype(int)
train_df_full['PRIMARY_RACE_White, Hispanic'] = train_df_full['PRIMARY_RACE_White, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_White, non-Hispanic'] = train_df_full['PRIMARY_RACE_White, non-Hispanic'].astype(int)

# Ethnicity
#train_df_full['ETHNICITY_0'] = train_df_full['ETHNICITY_0'].astype(int)
#train_df_full['ETHNICITY_Hispanic'] = train_df_full['ETHNICITY_Hispanic'].astype(int)
train_df_full['ETHNICITY_Hispanic/Latino'] = train_df_full['ETHNICITY_Hispanic/Latino'].astype(int)
train_df_full['ETHNICITY_Non-Hispanic/Non-Latino'] = train_df_full['ETHNICITY_Non-Hispanic/Non-Latino'].astype(int)
train_df_full['ETHNICITY_Not Hispanic'] = train_df_full['ETHNICITY_Not Hispanic'].astype(int)
train_df_full['ETHNICITY_Patient Refused'] = train_df_full['ETHNICITY_Patient Refused'].astype(int)

# Gender
train_df_full['GENDER_Male'] = train_df_full['GENDER_Male'].astype(int)
train_df_full['GENDER_Female'] = train_df_full['GENDER_Female'].astype(int)


In [8]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.spatial.distance import cosine  # <-- Added this line
import os

#Load the Data
csv_path = train_df_full
label_cols = ['Cardiomegaly', 'Edema', 'Consolidation', 'Pneumonia']
demographic_cols = ['Age', 'Sex']
df = csv_path

class TabularTimeSeriesDataset(Dataset):
    def __init__(self, df, label_cols, demographic_cols):
        self.label_cols = label_cols
        self.demographic_cols = demographic_cols

        if 'Age' in demographic_cols and 'Age' in df.columns:
            df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
            df['Age'] = (df['Age'] - df['Age'].mean()) / df['Age'].std()

        if 'Sex' in demographic_cols and 'Sex' in df.columns:
            df = pd.get_dummies(df, columns=['Sex'], prefix='Sex')
            self.demographic_cols = [col for col in df.columns if col.startswith('Sex_')] + (
                ['Age'] if 'Age' in df.columns else []
            )

        df.fillna(0, inplace=True)

        selected_cols = label_cols + self.demographic_cols
        self.data = df[selected_cols].copy()
        self.data = self.data.apply(pd.to_numeric, errors="coerce")
        self.data = self.data.fillna(0).astype(np.float32)

        self.mask = (~df[label_cols].isna()).astype(np.float32).values
        self.label_mask_len = len(label_cols)
        self.input_dim = self.data.shape[1]  # now correct

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = torch.tensor(self.data.iloc[idx].values, dtype=torch.float32)
        mask = torch.tensor(self.mask[idx], dtype=torch.float32)

        return x, mask[:self.label_mask_len]


    
# Split the dataframe
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_dataset = TabularTimeSeriesDataset(train_df, label_cols, demographic_cols)
val_dataset = TabularTimeSeriesDataset(val_df, label_cols, demographic_cols)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# DIFFUSION TRANSFORMER MODEL
class BayesianDiffusionTransformer(nn.Module):
    def __init__(self, input_dim, timesteps=1000):
        super().__init__()
        self.timesteps = timesteps
        self.input_dim = input_dim
        self.fc = nn.Sequential(
            nn.Linear(self.input_dim + 1, 128),  # <-- Fix here
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )
        self.beta = torch.linspace(1e-4, 0.02, timesteps)
        self.alpha = 1. - self.beta
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)

    def forward(self, x, t):
        assert x.shape[1] == self.input_dim, f"x has shape {x.shape}, expected input_dim={self.input_dim}"
        t = t.to(x.device).float() / self.timesteps
        t = t.unsqueeze(1)
        xt = torch.cat([x, t], dim=1)
        return self.fc(xt)



    def q_sample(self, x_start, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_start)
        alpha_hat_t = self.alpha_hat.to(x_start.device)[t].unsqueeze(1)
        return torch.sqrt(alpha_hat_t) * x_start + torch.sqrt(1 - alpha_hat_t) * noise

# Model directory
SAVE_DIR = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/Missing_values_Imputation_Beyesian_Diffusion_Transformer/"
os.makedirs(SAVE_DIR, exist_ok=True)

def save_checkpoint(model, optimizer, epoch, filename):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, filename)

def load_checkpoint(model, optimizer, filename, device='cpu'):
    checkpoint = torch.load(filename, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return checkpoint['epoch']

# ✅ MODIFIED TRAIN FUNCTION
def train(model, train_loader, val_loader, epochs=100, device='cpu', resume_path=None):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    start_epoch = 0

    resume_path = os.path.join(SAVE_DIR, resume_path) if resume_path else None
    if resume_path and os.path.exists(resume_path):
        print(f"Resuming training from {resume_path}")
        start_epoch = load_checkpoint(model, optimizer, resume_path, device)
        print(f"Resumed from epoch {start_epoch}")
    else:
        print(f"No checkpoint found at {resume_path}, starting from scratch.")

    for epoch in range(start_epoch, epochs):
        model.train()
        total_loss = 0

        for x, mask in train_loader:  # ✅ FIXED: used correct train_loader
            x = x.to(device)
            mask = mask.to(device)
            t = torch.randint(0, model.timesteps, (x.size(0),), device=device).long()
            noise = torch.randn_like(x)
            x_noisy = model.q_sample(x, t, noise)
            pred_noise = model(x_noisy, t)

            loss = ((pred_noise[:, :mask.shape[1]] - noise[:, :mask.shape[1]]) ** 2 * mask).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        model.eval()
        with torch.no_grad():
            val_preds = []
            val_targets = []

            for x_val, mask_val in val_loader:
                x_val = x_val.to(device)
                mask_val = mask_val.to(device)
                batch_size = x_val.size(0)
                t_val = torch.randint(0, model.timesteps, (batch_size,), device=device).long()

                noise_val = torch.randn_like(x_val)
                x_val_noisy = model.q_sample(x_val, t_val, noise_val)
                pred_val = model(x_val_noisy, t_val)

                val_preds.append(pred_val[:, :mask_val.shape[1]].cpu().numpy())
                val_targets.append(noise_val[:, :mask_val.shape[1]].cpu().numpy())

            val_preds = np.concatenate(val_preds, axis=0)
            val_targets = np.concatenate(val_targets, axis=0)

            val_mae = np.mean(np.abs(val_preds - val_targets))
            val_mse = np.mean((val_preds - val_targets) ** 2)
            val_rmse = np.sqrt(val_mse)
            val_r2 = r2_score(val_targets.flatten(), val_preds.flatten())

            print(f"Epoch {epoch + 1}")
            print(f"Validation MAE: {val_mae:.4f}, R²: {val_r2:.4f}, "
                f"MSE: {val_mse:.4f}, RMSE: {val_rmse:.4f}\n")


        checkpoint_path = os.path.join(SAVE_DIR, f"model_epoch_{epoch+1}.pt")
        save_checkpoint(model, optimizer, epoch+1, checkpoint_path)

# Initialize model and train
dataset = TabularTimeSeriesDataset(df, label_cols, demographic_cols)
input_dim = dataset.input_dim
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
model = BayesianDiffusionTransformer(input_dim=input_dim)
train(model, train_loader, val_loader)

No checkpoint found at None, starting from scratch.


AssertionError: x has shape torch.Size([32, 7]), expected input_dim=8

### What are we doing?

In [ ]:
# Project Overview
# You are designing a Bayesian Diffusion Transformer model to impute missing values in tabular medical data (e.g., disease labels and demographics) derived from the CheXpert dataset.

# 🎯 Key Research Objectives
# Impute Missing Clinical Labels

# Accurately fill in missing disease labels (e.g., Cardiomegaly, Edema) in chest X-ray datasets.

# Use a probabilistic approach to capture uncertainty.

# Leverage Bayesian Diffusion Models

# Use a forward diffusion process to corrupt data with noise.

# Train a Transformer-based denoising model to learn how to recover original signals.

# Handle Tabular + Time Series + Demographics

# Train on tabular data with columns like Age, Sex, and disease labels.

# Normalize, one-hot encode, and mask missing data properly for training.

# Ensure Demographic Fairness

# Ensure the model does not propagate demographic bias (e.g., over-imputing diseases based on Sex or Age).

# Enable Downstream Model Robustness

# Enable better generalization for disease classification models that use imputed data downstream.

# 🧪 Scientific Contributions
# ✅ Novel Architecture: Combines Diffusion Models (for uncertainty-aware modeling) with Transformers (for temporal/tabular dependencies).

# ✅ Masked Loss: Uses a masking strategy to only penalize the model on ground-truth (non-missing) labels.

# ✅ Synthetic Training Setup: Trains on partial information and evaluates how well it reconstructs the true data.

# ✅ Checkpoints + Resuming: Builds infrastructure for long training runs and experiment management.

#### Experiments in this work

In [ ]:
# ✅ Core Experiment Categories
# 1. Imputation Quality
# Metrics: MAE, RMSE, R² between true and imputed values (already done).

# Compare against:

# Mean/median imputation

# KNN imputation

# MICE (Multiple Imputation by Chained Equations)

# Variational Autoencoders (VAEs)

# TabNet or transformer-based baselines

# 2. Ablation Studies
# Remove components and compare performance:

# No diffusion noise (just a regular MLP)

# Without demographic features

# Replace diffusion steps with a simpler noise model

# Use fewer timesteps (e.g., 100, 500 instead of 1000)

# 3. Fairness Analysis
# Group-wise performance (MAE/R²) across:

# Age groups (e.g., 0–30, 31–50, etc.)

# Gender (e.g., male vs female)

# Race (if available)

# Permutation tests for fairness: Are errors significantly higher in one group?

# 4. Uncertainty Estimation (Bayesian Angle)
# Generate multiple samples per missing value.

# Report variance or confidence intervals for imputed values.

# Compare uncertainty-aware predictions with hard imputations.

# 5. Generalization Experiments
# Train on one hospital subset (or cohort), test on another (e.g., CheXpert → MIMIC).

# Hold out entire disease labels and test imputation in these settings.

# 6. Data Efficiency & Scaling
# Train on 10%, 25%, 50%, and 100% of the data.

# Plot MAE/R² vs dataset size to evaluate scaling behavior.

# 7. Synthetic-to-Real Evaluation
# Use your diffusion model to generate synthetic patient records.

# Train classifiers (e.g., XGBoost) on imputed vs synthetic vs original data.

# Evaluate downstream disease prediction accuracy.